# Week 10: Fixed Effects Models

**QM 2023 — Statistics II / Data Analytics**
**Spring 2026 | University of Tulsa**

---

## What You Will Learn

By the end of this demo, you will be able to:

1. **Explain** why pooled OLS is biased when unobserved entity characteristics exist
2. **Distinguish** between "within" and "between" variation in panel data
3. **Estimate** Entity Fixed Effects models using `linearmodels.PanelOLS`
4. **Estimate** Two-Way Fixed Effects models (entity + time effects)
5. **Compare** coefficients across Pooled OLS, Entity FE, and Two-Way FE specifications
6. **Interpret** the F-test for joint significance of fixed effects
7. **Explain** why R-squared may drop when moving from Pooled OLS to Fixed Effects

---

### The Big Idea

Last week, we learned how to structure **panel data** — the same entities observed over time. This week, we use that structure to solve a fundamental problem in causal inference: **omitted variable bias**.

The key insight: if unobserved characteristics (like management quality or brand value) are correlated with both our predictors and our outcome, then pooled OLS gives us **biased** coefficients. Fixed Effects solves this by giving each entity its own intercept, effectively "controlling for" everything that is constant within an entity over time.

$$Y_{it} = \beta X_{it} + \alpha_i + \epsilon_{it}$$

where $\alpha_i$ absorbs all time-invariant characteristics of entity $i$.

---

## Section 1: Setup and Data Loading

We need these libraries:

- **pandas / numpy**: Data manipulation
- **matplotlib**: Visualizations
- **statsmodels**: Pooled OLS estimation
- **linearmodels**: Panel data models (PanelOLS for Fixed Effects)

In [ ]:
# Standard data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Panel data models
from linearmodels.panel import PanelOLS

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Plot style settings for consistent, beginner-friendly visuals
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 10

# Consistent color palette used throughout this notebook
COLOR_POOLED = '#c44e52'       # Red for Pooled OLS (the "biased" model)
COLOR_ENTITY_FE = '#4c72b0'    # Blue for Entity FE
COLOR_TWOWAY_FE = '#55a868'    # Green for Two-Way FE
COLOR_NEUTRAL = '#6c757d'      # Gray for neutral elements
COLOR_ACCENT = '#8172b3'       # Purple for accents

# Path to demo output folder (for saving figures)
from pathlib import Path
DEMO_DIR = Path('.')  # Figures saved alongside this notebook

print("Imports successful!")
print(f"  pandas {pd.__version__}, numpy {np.__version__}")

### Load the REIT Panel Data

We use real REIT (Real Estate Investment Trust) data. Each row is one REIT in one year. The key variables we will focus on:

| Variable | Description |
|----------|-------------|
| `permno` | Unique REIT identifier (entity) |
| `year` | Calendar year (time) |
| `ret12` | Annual stock return (our **outcome**) |
| `lnmcap` | Log of market capitalization (a **predictor** — firm size) |
| `beta` | Market risk exposure (a **predictor**) |
| `bookleverage` | Financial leverage ratio (a **predictor**) |

In [ ]:
# Load the REIT panel data from this workspace
reit_data_path = Path('data/REIT_sample_annual_2004_2024.csv')

if not reit_data_path.exists():
    raise FileNotFoundError('Could not find data/REIT_sample_annual_2004_2024.csv in this repository.')

df_raw = pd.read_csv(reit_data_path)

print(f"Loaded data from: {reit_data_path.resolve()}")
print(f"Raw data shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\nFirst 5 rows:")
df_raw.head()

In [ ]:
# Select the columns we need and drop rows with missing values
columns_to_keep = ['permno', 'year', 'ticker', 'comnam', 'ret12', 'lnmcap', 'beta', 'bookleverage']
df = df_raw[columns_to_keep].dropna().copy()

# Check the panel dimensions
n_reits = df['permno'].nunique()       # Number of unique REITs (entities)
n_years = df['year'].nunique()          # Number of unique years (time periods)
n_observations = len(df)               # Total observations

print(f"Panel Dimensions:")
print(f"  N (unique REITs):    {n_reits}")
print(f"  T (unique years):    {n_years}")
print(f"  Total observations:  {n_observations}")
print(f"  Year range:          {df['year'].min()} - {df['year'].max()}")
print(f"\nDescriptive statistics for key variables:")
df[['ret12', 'lnmcap', 'beta', 'bookleverage']].describe().round(3)

In [ ]:
# Chart 1: Number of REITs observed per year (bar chart)
reits_per_year = df.groupby('year')['permno'].nunique()  # Count unique REITs in each year

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(reits_per_year.index, reits_per_year.values, color=COLOR_ENTITY_FE, alpha=0.8, edgecolor='white')
ax.set_xlabel('Year')
ax.set_ylabel('Number of REITs')
ax.set_title('Number of REITs Observed Per Year')

# Add count labels on top of each bar
for year_val, count_val in zip(reits_per_year.index, reits_per_year.values):
    ax.text(year_val, count_val + 0.5, str(count_val), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_reits_per_year.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_reits_per_year.png")
plt.show()

### Interpretation: Panel Structure

This is an **unbalanced panel** — the number of REITs varies from year to year because firms enter and exit the sample. That is completely normal in real-world data. The important thing is that we observe *the same entities* across *multiple time periods*, which gives us the "within-entity" variation we need for Fixed Effects.

---

## Section 2: Visualizing the Problem — Omitted Variable Bias

Before we estimate any models, let's **see** the problem that Fixed Effects is designed to solve.

**The question:** Does firm size (`lnmcap`) predict annual returns (`ret12`)?

If we just throw all the data into a scatter plot (pooling across all REITs), we are mixing up two very different sources of variation:

1. **Between-entity variation:** Some REITs are systematically larger than others
2. **Within-entity variation:** A given REIT's size changes over time

Pooled OLS combines both. Fixed Effects uses only #2.

In [ ]:
# Chart 2: Pooled scatter plot — lnmcap vs ret12 (all REITs lumped together)
annual_return = df['ret12']          # Outcome: annual stock return
log_market_cap = df['lnmcap']       # Predictor: log market capitalization

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(log_market_cap, annual_return, alpha=0.15, s=15, color=COLOR_NEUTRAL, label='All REIT-year observations')
ax.set_xlabel('Log Market Cap (lnmcap)')
ax.set_ylabel('Annual Return (ret12)')
ax.set_title('Pooled Scatter: Firm Size vs. Annual Return\n(All REITs, All Years — Ignoring Panel Structure)')
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_pooled_scatter.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_pooled_scatter.png")
plt.show()

That scatter plot treats every observation as independent. But we know that the same REIT appears multiple times! Let's highlight a few specific REITs to see the **entity heterogeneity** — different REITs have different baseline levels.

In [ ]:
# Chart 3: Highlight a few specific REITs to show entity heterogeneity
# Pick 5 REITs that appear in many years (so we can see their trajectories)
reit_year_counts = df.groupby('permno')['year'].count()               # How many years each REIT appears
reits_with_many_years = reit_year_counts[reit_year_counts >= 10].index  # REITs with 10+ observations
highlight_reits = list(reits_with_many_years[:5])                      # Pick the first 5

# Get ticker names for the legend
reit_tickers = df[df['permno'].isin(highlight_reits)].groupby('permno')['ticker'].first()

# Color palette for highlighted REITs
highlight_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

fig, ax = plt.subplots(figsize=(10, 6))

# Plot all data points in gray (background)
ax.scatter(df['lnmcap'], df['ret12'], alpha=0.08, s=10, color=COLOR_NEUTRAL, label='Other REITs')

# Overlay the highlighted REITs in color
for i, reit_id in enumerate(highlight_reits):
    reit_subset = df[df['permno'] == reit_id]                           # Filter to this REIT
    reit_lnmcap = reit_subset['lnmcap']                                # This REIT's log market cap
    reit_return = reit_subset['ret12']                                  # This REIT's annual return
    reit_ticker = reit_tickers[reit_id]                                 # This REIT's ticker symbol
    ax.scatter(reit_lnmcap, reit_return, alpha=0.8, s=40, 
               color=highlight_colors[i], label=f'{reit_ticker} (permno={reit_id})', 
               edgecolors='black', linewidth=0.3)

ax.set_xlabel('Log Market Cap (lnmcap)')
ax.set_ylabel('Annual Return (ret12)')
ax.set_title('Entity Heterogeneity: Different REITs Occupy Different Regions\n(Each color = one REIT observed across multiple years)')
ax.legend(loc='upper right', fontsize=9)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_entity_heterogeneity.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_entity_heterogeneity.png")
plt.show()

### What do you see?

Each colored cluster represents **one REIT** observed across multiple years. Notice how different REITs "live" in different parts of the plot — some are consistently large, others small; some have higher average returns, others lower.

This is **entity heterogeneity**: unobserved REIT-specific characteristics (management quality, property portfolio, geographic focus) that affect both size and returns.

**The problem:** If we run a single regression line through all the gray dots, we are mixing up "REITs that are bigger tend to have different returns" (between variation) with "when a REIT gets bigger, its return changes" (within variation). These are **different questions** and may have **different answers**.

In [ ]:
# Chart 4: Spaghetti plot — individual REIT return trajectories over time
# This shows how different REITs follow different paths

fig, ax = plt.subplots(figsize=(12, 6))

# Plot trajectories for REITs with at least 8 years of data (for readability)
reits_with_history = df.groupby('permno').filter(lambda x: len(x) >= 8)['permno'].unique()

# Plot each REIT as a faint line
for reit_id in reits_with_history:
    reit_subset = df[df['permno'] == reit_id].sort_values('year')
    ax.plot(reit_subset['year'], reit_subset['ret12'], alpha=0.12, linewidth=0.8, color=COLOR_NEUTRAL)

# Overlay the 5 highlighted REITs in bold colors
for i, reit_id in enumerate(highlight_reits):
    reit_subset = df[df['permno'] == reit_id].sort_values('year')
    reit_ticker = reit_tickers[reit_id]
    ax.plot(reit_subset['year'], reit_subset['ret12'], alpha=0.9, linewidth=2.5, 
            color=highlight_colors[i], label=f'{reit_ticker}', marker='o', markersize=4)

ax.set_xlabel('Year')
ax.set_ylabel('Annual Return (ret12)')
ax.set_title('Individual REIT Return Trajectories Over Time\n(Each line = one REIT; colored lines = highlighted REITs)')
ax.legend(loc='upper right', fontsize=9)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_spaghetti.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_spaghetti.png")
plt.show()

---

## Section 3: Within vs. Between Variation

This is the **core concept** of Fixed Effects. All variation in our data can be split into two parts:

- **Between variation**: How does REIT A's *average* differ from REIT B's *average*? (Cross-sectional differences)
- **Within variation**: How does REIT A deviate from its *own* average over time? (Temporal changes for the same entity)

**Fixed Effects throws away the between variation and uses only within variation.** Why? Because between-entity differences might be driven by unobserved characteristics (management quality, location, etc.) that confound our estimate.

In [ ]:
# Calculate group means for each REIT (the "between" component)
reit_means = df.groupby('permno')[['ret12', 'lnmcap', 'beta', 'bookleverage']].mean()
reit_means.columns = ['mean_ret12', 'mean_lnmcap', 'mean_beta', 'mean_bookleverage']

# Merge the group means back onto the original data
df = df.merge(reit_means, left_on='permno', right_index=True, how='left')

# Calculate the "within" component: deviation from group mean
df['within_ret12'] = df['ret12'] - df['mean_ret12']           # How much this year's return deviates from the REIT's average
df['within_lnmcap'] = df['lnmcap'] - df['mean_lnmcap']       # How much this year's size deviates from the REIT's average

print("Variation decomposition for log market cap (lnmcap):")
total_var_lnmcap = df['lnmcap'].var()                          # Total variance
between_var_lnmcap = df['mean_lnmcap'].var()                   # Between-entity variance (variance of entity means)
within_var_lnmcap = df['within_lnmcap'].var()                  # Within-entity variance (variance of deviations)

print(f"  Total variance:    {total_var_lnmcap:.4f}")
print(f"  Between (across REITs): {between_var_lnmcap:.4f}  ({between_var_lnmcap/total_var_lnmcap*100:.1f}%)")
print(f"  Within (over time):     {within_var_lnmcap:.4f}  ({within_var_lnmcap/total_var_lnmcap*100:.1f}%)")

print(f"\nVariation decomposition for annual return (ret12):")
total_var_ret12 = df['ret12'].var()
between_var_ret12 = df['mean_ret12'].var()
within_var_ret12 = df['within_ret12'].var()

print(f"  Total variance:    {total_var_ret12:.4f}")
print(f"  Between (across REITs): {between_var_ret12:.4f}  ({between_var_ret12/total_var_ret12*100:.1f}%)")
print(f"  Within (over time):     {within_var_ret12:.4f}  ({within_var_ret12/total_var_ret12*100:.1f}%)")

In [ ]:
# Chart 5: Between variation (scatter of REIT means) vs. Within variation (deviations)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Between variation — each dot is one REIT's average
between_lnmcap = reit_means['mean_lnmcap']        # Each REIT's average log market cap
between_ret12 = reit_means['mean_ret12']            # Each REIT's average annual return

axes[0].scatter(between_lnmcap, between_ret12, alpha=0.6, s=30, color=COLOR_POOLED, edgecolors='black', linewidth=0.3)
axes[0].set_xlabel('REIT Mean Log Market Cap')
axes[0].set_ylabel('REIT Mean Annual Return')
axes[0].set_title('BETWEEN Variation\n(Each dot = one REIT\'s time average)')
axes[0].axhline(y=0, color='black', linewidth=0.5, linestyle='-')

# Right panel: Within variation — deviations from REIT means
within_lnmcap = df['within_lnmcap']               # Deviation of lnmcap from REIT mean
within_ret12 = df['within_ret12']                   # Deviation of ret12 from REIT mean

axes[1].scatter(within_lnmcap, within_ret12, alpha=0.15, s=10, color=COLOR_ENTITY_FE)
axes[1].set_xlabel('Within-REIT Deviation in Log Market Cap')
axes[1].set_ylabel('Within-REIT Deviation in Annual Return')
axes[1].set_title('WITHIN Variation\n(Each dot = one REIT-year deviation from its own mean)')
axes[1].axhline(y=0, color='black', linewidth=0.5, linestyle='-')
axes[1].axvline(x=0, color='black', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_within_between.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_within_between.png")
plt.show()

### Interpretation: Within vs. Between

- **Left panel (Between):** Each dot is one REIT's time-average. The relationship here reflects *which REITs* are bigger and *which REITs* have higher returns. This mixes in all the unobserved entity characteristics.

- **Right panel (Within):** Each dot shows how a REIT-year *deviates from that REIT's own average*. The relationship here answers: "When a REIT gets bigger than its own norm, does its return also deviate?"

**Fixed Effects uses only the right panel.** By "demeaning" — subtracting each REIT's average — we remove all time-invariant confounders. This is mathematically equivalent to adding a separate intercept (dummy variable) for every REIT.

---

## Section 4: Model 1 — Pooled OLS

Our first model ignores the panel structure entirely. It treats every REIT-year observation as independent and estimates a single regression line:

$$\text{ret12}_{it} = \beta_0 + \beta_1 \cdot \text{lnmcap}_{it} + \beta_2 \cdot \text{beta}_{it} + \varepsilon_{it}$$

This is the "naive" approach. If unobserved REIT characteristics are correlated with our predictors, these coefficients will be **biased**.

In [ ]:
# Model 1: Pooled OLS — ignores panel structure

# Define the outcome variable
y_annual_return = df['ret12']                           # Outcome: annual stock return

# Define the predictor variables
X_predictors = df[['lnmcap', 'beta']]                  # Predictors: log market cap and market beta
X_design_pooled = sm.add_constant(X_predictors)         # Add intercept column

# Estimate Pooled OLS
pooled_ols_model = sm.OLS(y_annual_return, X_design_pooled).fit()

print("=" * 60)
print("MODEL 1: POOLED OLS (ignores panel structure)")
print("=" * 60)
print(pooled_ols_model.summary().tables[1])

# Extract key results for comparison later
pooled_lnmcap_coef = pooled_ols_model.params['lnmcap']       # Coefficient on log market cap
pooled_beta_coef = pooled_ols_model.params['beta']             # Coefficient on market beta
pooled_r_squared = pooled_ols_model.rsquared                   # R-squared

print(f"\nKey Results:")
print(f"  lnmcap coefficient: {pooled_lnmcap_coef:.4f}")
print(f"  beta coefficient:   {pooled_beta_coef:.4f}")
print(f"  R-squared:          {pooled_r_squared:.4f}")

In [ ]:
# Chart 6: Pooled OLS fitted line overlaid on scatter
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter all data
ax.scatter(df['lnmcap'], df['ret12'], alpha=0.12, s=10, color=COLOR_NEUTRAL, label='Observations')

# Fitted line: predict ret12 at beta=mean(beta), varying lnmcap
lnmcap_range = np.linspace(df['lnmcap'].min(), df['lnmcap'].max(), 100)
beta_mean = df['beta'].mean()                                  # Hold beta at its mean
predicted_return_pooled = (pooled_ols_model.params['const']
                           + pooled_ols_model.params['lnmcap'] * lnmcap_range
                           + pooled_ols_model.params['beta'] * beta_mean)

ax.plot(lnmcap_range, predicted_return_pooled, color=COLOR_POOLED, linewidth=3,
        label=f'Pooled OLS (slope = {pooled_lnmcap_coef:.4f})')

ax.set_xlabel('Log Market Cap (lnmcap)')
ax.set_ylabel('Annual Return (ret12)')
ax.set_title('Pooled OLS: One Line for All REITs\n(Ignores entity-specific differences)')
ax.legend(fontsize=10)
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='-')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_pooled_ols_fit.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_pooled_ols_fit.png")
plt.show()

### Pooled OLS Interpretation

The Pooled OLS coefficient on `lnmcap` tells us the relationship between firm size and returns **across all REITs and all years combined**. But this coefficient reflects a mixture of:

- Between-entity: "Are larger REITs different from smaller REITs?"
- Within-entity: "When a REIT grows, does its return change?"

If management quality (unobserved) is correlated with both size and returns, the Pooled OLS coefficient is **biased**. Fixed Effects will help us isolate the within-entity relationship.

---

## Section 5: Model 2 — Entity Fixed Effects (One-Way FE)

Now we add a separate intercept for each REIT. This absorbs all **time-invariant** REIT characteristics:

$$\text{ret12}_{it} = \alpha_i + \beta_1 \cdot \text{lnmcap}_{it} + \beta_2 \cdot \text{beta}_{it} + \varepsilon_{it}$$

where $\alpha_i$ is a REIT-specific intercept. We estimate this using `linearmodels.PanelOLS`.

**Critical step:** We must set a MultiIndex `(permno, year)` before calling PanelOLS.

In [ ]:
# CRITICAL: Create MultiIndex for panel data
# PanelOLS requires the DataFrame to have a MultiIndex with (entity, time)
panel_df = df.set_index(['permno', 'year']).sort_index()

print("MultiIndex structure:")
print(f"  Index names:  {panel_df.index.names}")
print(f"  Index levels: {len(panel_df.index.get_level_values(0).unique())} entities x "
      f"{len(panel_df.index.get_level_values(1).unique())} time periods")
print(f"\nFirst 5 rows of panel_df:")
panel_df[['ret12', 'lnmcap', 'beta']].head()

In [ ]:
# Model 2: Entity Fixed Effects using PanelOLS

# Define dependent variable (outcome)
y_panel_return = panel_df['ret12']

# Define independent variables (predictors) — no constant needed, FE absorbs it
X_panel_predictors = panel_df[['lnmcap', 'beta']]

# Estimate Entity FE model
entity_fe_model = PanelOLS(y_panel_return, X_panel_predictors, entity_effects=True)
entity_fe_results = entity_fe_model.fit(cov_type='clustered', cluster_entity=True)

print("=" * 60)
print("MODEL 2: ENTITY FIXED EFFECTS")
print("=" * 60)
print(entity_fe_results.summary.tables[1])

# Extract key results
entity_fe_lnmcap_coef = entity_fe_results.params['lnmcap']
entity_fe_beta_coef = entity_fe_results.params['beta']
entity_fe_r_squared = entity_fe_results.rsquared

print(f"\nKey Results:")
print(f"  lnmcap coefficient: {entity_fe_lnmcap_coef:.4f}")
print(f"  beta coefficient:   {entity_fe_beta_coef:.4f}")
print(f"  R-squared (within): {entity_fe_r_squared:.4f}")

In [ ]:
# Chart 7: Coefficient comparison — Pooled OLS vs Entity FE
fig, ax = plt.subplots(figsize=(8, 5))

variables = ['lnmcap', 'beta']
x_positions = np.arange(len(variables))
bar_width = 0.35

pooled_coefs = [pooled_lnmcap_coef, pooled_beta_coef]
entity_fe_coefs = [entity_fe_lnmcap_coef, entity_fe_beta_coef]

# Confidence intervals for error bars
pooled_lnmcap_ci = pooled_ols_model.conf_int().loc['lnmcap']
pooled_beta_ci = pooled_ols_model.conf_int().loc['beta']
pooled_errors = [
    [pooled_lnmcap_coef - pooled_lnmcap_ci[0], pooled_beta_coef - pooled_beta_ci[0]],
    [pooled_lnmcap_ci[1] - pooled_lnmcap_coef, pooled_beta_ci[1] - pooled_beta_coef]
]

fe_ci = entity_fe_results.conf_int()
fe_errors = [
    [entity_fe_lnmcap_coef - fe_ci.loc['lnmcap', 'lower'],
     entity_fe_beta_coef - fe_ci.loc['beta', 'lower']],
    [fe_ci.loc['lnmcap', 'upper'] - entity_fe_lnmcap_coef,
     fe_ci.loc['beta', 'upper'] - entity_fe_beta_coef]
]

ax.bar(x_positions - bar_width/2, pooled_coefs, bar_width,
       yerr=pooled_errors, capsize=5,
       label='Pooled OLS', color=COLOR_POOLED, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.bar(x_positions + bar_width/2, entity_fe_coefs, bar_width,
       yerr=fe_errors, capsize=5,
       label='Entity FE', color=COLOR_ENTITY_FE, alpha=0.8, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Variable')
ax.set_ylabel('Coefficient Estimate')
ax.set_title('Coefficient Comparison: Pooled OLS vs. Entity FE\n(Error bars = 95% confidence intervals)')
ax.set_xticks(x_positions)
ax.set_xticklabels(variables)
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_coef_comparison_2model.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_coef_comparison_2model.png")
plt.show()

### What Changed?

Compare the `lnmcap` coefficient between Pooled OLS and Entity FE:

- **Pooled OLS** includes between-entity variation — it reflects *which* REITs are bigger
- **Entity FE** uses only within-entity variation — it reflects what happens *when a given REIT changes size*

The difference between these two coefficients is evidence of **omitted variable bias** in the Pooled OLS model.

---

## Section 6: Model 3 — Two-Way Fixed Effects (Entity + Time)

Entity FE controls for REIT-specific characteristics, but what about market-wide shocks that affect *all* REITs in a given year (e.g., the 2008 financial crisis, the 2020 pandemic)?

Two-Way FE adds **time fixed effects** ($\gamma_t$) on top of entity fixed effects:

$$\text{ret12}_{it} = \alpha_i + \gamma_t + \beta_1 \cdot \text{lnmcap}_{it} + \beta_2 \cdot \text{beta}_{it} + \varepsilon_{it}$$

This absorbs both entity-specific *and* year-specific unobserved factors.

In [ ]:
# Model 3: Two-Way Fixed Effects (Entity + Time)
y_panel_return = panel_df['ret12']
X_panel_predictors = panel_df[['lnmcap', 'beta']]

# entity_effects=True AND time_effects=True for two-way FE
twoway_fe_model = PanelOLS(
    y_panel_return, X_panel_predictors,
    entity_effects=True, time_effects=True
)
twoway_fe_results = twoway_fe_model.fit(cov_type='clustered', cluster_entity=True)

print("=" * 60)
print("MODEL 3: TWO-WAY FIXED EFFECTS (Entity + Time)")
print("=" * 60)
print(twoway_fe_results.summary.tables[1])

# Extract key results
twoway_lnmcap_coef = twoway_fe_results.params['lnmcap']
twoway_beta_coef = twoway_fe_results.params['beta']
twoway_r_squared = twoway_fe_results.rsquared

print(f"\nKey Results:")
print(f"  lnmcap coefficient: {twoway_lnmcap_coef:.4f}")
print(f"  beta coefficient:   {twoway_beta_coef:.4f}")
print(f"  R-squared (within): {twoway_r_squared:.4f}")

---

## Section 7: F-Test — Are the Fixed Effects Significant?

We need to test whether the entity fixed effects are **jointly significant**. The null hypothesis is:

$$H_0: \alpha_1 = \alpha_2 = \ldots = \alpha_N = 0$$

A large F-statistic means the entity effects are jointly significant — we **need** them.

In [ ]:
# F-test for joint significance of entity fixed effects
entity_f_stat = entity_fe_results.f_statistic

print("=" * 60)
print("F-TEST: Are Entity Fixed Effects Jointly Significant?")
print("=" * 60)
print(f"  F-statistic:  {entity_f_stat.stat:.2f}")
print(f"  p-value:      {entity_f_stat.pval:.6f}")
print(f"  df (num):     {entity_f_stat.df:.0f}")
print(f"  df (denom):   {entity_f_stat.df_denom:.0f}")

if entity_f_stat.pval < 0.05:
    print(f"\n  CONCLUSION: Reject H0 at 5% level.")
    print(f"  The entity fixed effects ARE jointly significant.")
    print(f"  REITs have meaningfully different baseline characteristics.")
    print(f"  Pooled OLS is inappropriate for this data.")
else:
    print(f"\n  CONCLUSION: Fail to reject H0.")
    print(f"  Entity fixed effects may not be needed.")

---

## Section 8: Three-Model Comparison Table

Now let's put all three models side by side to see how the coefficients change as we add more controls for unobserved heterogeneity.

In [ ]:
# Build comparison table
comparison_data = {
    'Pooled OLS': {
        'lnmcap': f"{pooled_lnmcap_coef:.4f}",
        'beta': f"{pooled_beta_coef:.4f}",
        'R-squared': f"{pooled_r_squared:.4f}",
        'Entity FE': 'No',
        'Time FE': 'No',
        'Observations': str(int(pooled_ols_model.nobs)),
    },
    'Entity FE': {
        'lnmcap': f"{entity_fe_lnmcap_coef:.4f}",
        'beta': f"{entity_fe_beta_coef:.4f}",
        'R-squared': f"{entity_fe_r_squared:.4f}",
        'Entity FE': 'Yes',
        'Time FE': 'No',
        'Observations': str(entity_fe_results.nobs),
    },
    'Two-Way FE': {
        'lnmcap': f"{twoway_lnmcap_coef:.4f}",
        'beta': f"{twoway_beta_coef:.4f}",
        'R-squared': f"{twoway_r_squared:.4f}",
        'Entity FE': 'Yes',
        'Time FE': 'Yes',
        'Observations': str(twoway_fe_results.nobs),
    }
}

comparison_table = pd.DataFrame(comparison_data)
print("=" * 60)
print("THREE-MODEL COMPARISON TABLE")
print("=" * 60)
print(comparison_table.to_string())

---

## Section 9: Visual Summary

In [ ]:
# Chart 8: Three-model coefficient comparison (dot plot with CIs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ['Pooled OLS', 'Entity FE', 'Two-Way FE']
colors = [COLOR_POOLED, COLOR_ENTITY_FE, COLOR_TWOWAY_FE]
y_positions = [2, 1, 0]

# --- Left panel: lnmcap coefficient ---
lnmcap_coefs = [pooled_lnmcap_coef, entity_fe_lnmcap_coef, twoway_lnmcap_coef]

pooled_ci_lnmcap = pooled_ols_model.conf_int().loc['lnmcap']
fe_ci_lnmcap = entity_fe_results.conf_int().loc['lnmcap']
tw_ci_lnmcap = twoway_fe_results.conf_int().loc['lnmcap']

lnmcap_lower = [pooled_ci_lnmcap[0], fe_ci_lnmcap['lower'], tw_ci_lnmcap['lower']]
lnmcap_upper = [pooled_ci_lnmcap[1], fe_ci_lnmcap['upper'], tw_ci_lnmcap['upper']]

for i in range(3):
    axes[0].plot([lnmcap_lower[i], lnmcap_upper[i]], [y_positions[i], y_positions[i]],
                 color=colors[i], linewidth=2.5, solid_capstyle='round')
    axes[0].scatter([lnmcap_coefs[i]], [y_positions[i]],
                    color=colors[i], s=100, zorder=5, edgecolors='black', linewidth=0.5)
    axes[0].text(lnmcap_upper[i] + 0.002, y_positions[i],
                 f"{lnmcap_coefs[i]:.4f}", va='center', fontsize=10)

axes[0].axvline(x=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(models)
axes[0].set_xlabel('Coefficient on lnmcap')
axes[0].set_title('lnmcap Coefficient Across Models\n(with 95% CI)')

# --- Right panel: beta coefficient ---
beta_coefs = [pooled_beta_coef, entity_fe_beta_coef, twoway_beta_coef]

pooled_ci_beta = pooled_ols_model.conf_int().loc['beta']
fe_ci_beta = entity_fe_results.conf_int().loc['beta']
tw_ci_beta = twoway_fe_results.conf_int().loc['beta']

beta_lower = [pooled_ci_beta[0], fe_ci_beta['lower'], tw_ci_beta['lower']]
beta_upper = [pooled_ci_beta[1], fe_ci_beta['upper'], tw_ci_beta['upper']]

for i in range(3):
    axes[1].plot([beta_lower[i], beta_upper[i]], [y_positions[i], y_positions[i]],
                 color=colors[i], linewidth=2.5, solid_capstyle='round')
    axes[1].scatter([beta_coefs[i]], [y_positions[i]],
                    color=colors[i], s=100, zorder=5, edgecolors='black', linewidth=0.5)
    axes[1].text(beta_upper[i] + 0.005, y_positions[i],
                 f"{beta_coefs[i]:.4f}", va='center', fontsize=10)

axes[1].axvline(x=0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_yticks(y_positions)
axes[1].set_yticklabels(models)
axes[1].set_xlabel('Coefficient on beta')
axes[1].set_title('beta Coefficient Across Models\n(with 95% CI)')

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_coef_comparison_3model.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_coef_comparison_3model.png")
plt.show()

In [ ]:
# Chart 9: R-squared comparison across models
fig, ax = plt.subplots(figsize=(8, 5))

model_names = ['Pooled OLS', 'Entity FE', 'Two-Way FE']
r_squared_values = [pooled_r_squared, entity_fe_r_squared, twoway_r_squared]
bar_colors = [COLOR_POOLED, COLOR_ENTITY_FE, COLOR_TWOWAY_FE]

bars = ax.bar(model_names, r_squared_values, color=bar_colors, alpha=0.8,
              edgecolor='black', linewidth=0.5)

# Add value labels on each bar
for bar, r2 in zip(bars, r_squared_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{r2:.4f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('R-squared')
ax.set_title('R-squared Comparison Across Models\n'
             '(Lower FE R-squared is expected — between variation is absorbed)')
ax.set_ylim(0, max(r_squared_values) * 1.3)

plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_r_squared_comparison.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_r_squared_comparison.png")
plt.show()

### Why Does R-squared Drop?

You might expect that adding more controls would **increase** R-squared. But FE R-squared often **drops** compared to Pooled OLS. Why?

- **Pooled OLS R-squared** reflects how well the model explains *total* variation (between + within)
- **FE R-squared** reflects how well the model explains *within-entity* variation only (after demeaning)

The between-entity variation is **absorbed by the fixed effects** — it is no longer in the residual or the "explained" part that R-squared measures. A lower R-squared with FE does **not** mean a worse model. The goal is **unbiased coefficients**, not high R-squared.

In [ ]:
# Chart 10: Residual distributions across models
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# Pooled OLS residuals
pooled_residuals = pooled_ols_model.resid
axes[0].hist(pooled_residuals, bins=50, color=COLOR_POOLED, alpha=0.7, edgecolor='white')
axes[0].set_title(f'Pooled OLS Residuals\n(std = {pooled_residuals.std():.3f})')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].axvline(x=0, color='black', linewidth=0.8, linestyle='--')

# Entity FE residuals
entity_fe_residuals = entity_fe_results.resids
axes[1].hist(entity_fe_residuals, bins=50, color=COLOR_ENTITY_FE, alpha=0.7, edgecolor='white')
axes[1].set_title(f'Entity FE Residuals\n(std = {entity_fe_residuals.std():.3f})')
axes[1].set_xlabel('Residual')
axes[1].axvline(x=0, color='black', linewidth=0.8, linestyle='--')

# Two-Way FE residuals
twoway_fe_residuals = twoway_fe_results.resids
axes[2].hist(twoway_fe_residuals, bins=50, color=COLOR_TWOWAY_FE, alpha=0.7, edgecolor='white')
axes[2].set_title(f'Two-Way FE Residuals\n(std = {twoway_fe_residuals.std():.3f})')
axes[2].set_xlabel('Residual')
axes[2].axvline(x=0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('Residual Distributions Across Models', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(DEMO_DIR / 'fe_demo_residual_distributions.png', dpi=150, bbox_inches='tight')
print("Saved: fe_demo_residual_distributions.png")
plt.show()

---

## Section 10: Key Takeaways

1. **Pooled OLS is biased** when unobserved entity characteristics are correlated with predictors. The coefficients mix between-entity and within-entity variation.

2. **Entity Fixed Effects** controls for all time-invariant entity characteristics by giving each REIT its own intercept. Coefficients now reflect *within-entity* changes only.

3. **Two-Way Fixed Effects** adds time dummies to also absorb year-specific shocks (e.g., 2008 crisis, 2020 pandemic) that affect all entities simultaneously.

4. **The F-test** tells us whether the entity fixed effects are jointly significant. A significant F-test means Pooled OLS is inappropriate.

5. **R-squared may drop** from Pooled OLS to FE — this is expected and does not indicate a worse model. Between-entity variation is absorbed, not lost.

6. **Key Python steps**:

   - Set MultiIndex: `df.set_index(['permno', 'year']).sort_index()`
   - Entity FE: `PanelOLS(y, X, entity_effects=True)`
   - Two-Way FE: `PanelOLS(y, X, entity_effects=True, time_effects=True)`
   - Clustered SEs: `.fit(cov_type='clustered', cluster_entity=True)`

---

**Next week:** We use Fixed Effects as a building block for **Difference-in-Differences** — the gold standard for evaluating policy changes and natural experiments.